# English-to-Yoda Style Text Generation

This notebook implements a small generative text-style transfer project. The goal is to fine-tune pretrained sequence-to-sequence models so that they transform normal English sentences into Yoda-style sentences!
The project uses the Hugging Face dataset `dvgodoy/yoda_sentences`, which contains pairs of normal English sentences and their Yoda-style versions. Two models are compared: `T5-small` as a baseline model and `FLAN-T5-small` as a more recent instruction-tuned alternative. Both models are trained with the same dataset split and hyperparameters so their performance can be compared fairly.

# 1. Install Required Libraries

In [ ]:
!pip install transformers datasets evaluate sacrebleu accelerate -q

# 2. Import Libraries and Set the Random Seed

In [ ]:
import gc
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)

# here we set a fixed random seed so that the dataset split and training behavior are more reproducible
set_seed(42)


# 3. Load the Yoda Sentences Dataset from Hugging Face

In [ ]:
dataset = load_dataset("dvgodoy/yoda_sentences")
dataset

# 4. Inspect One Example and the Dataset Columns

This cell prints the first dataset example and the available column names.

In [ ]:
dataset["train"][0], dataset["train"].column_names

# 5. Split the Dataset into Training and Validation Sets

This cell splits the dataset into a training set and a validation set.


In [ ]:
dataset_split = dataset["train"].train_test_split(test_size=0.2, seed=42)

train_dataset = dataset_split["train"]
val_dataset = dataset_split["test"]

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

# 6. Fine-Tune and Compare T5-small and FLAN-T5-small

This cell contains the main fine-tuning pipeline. It defines two models, prepares the English-to-Yoda-style input/output format, tokenizes the data, fine-tunes each model for the same number of epochs, evaluates after each epoch, and stores the training and validation losses for comparison.

The goal is to compare the older `T5-small` model with the more recent instruction-tuned `FLAN-T5-small` model using the same dataset and training setup.


In [ ]:
source_column = "sentence"
target_column = "translation_extra"
max_input_length = 64
max_target_length = 64


def train_model(model_label, model_name):
    print(f"\nTraining {model_label}: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def preprocess_function(examples):
        inputs = [
            "translate English to Yoda style: " + text
            for text in examples[source_column]
        ]

        model_inputs = tokenizer(
            inputs,
            max_length=max_input_length,
            truncation=True,
        )

        labels = tokenizer(
            text_target=examples[target_column],
            max_length=max_target_length,
            truncation=True,
        )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized_train = train_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=train_dataset.column_names,
    )

    tokenized_val = val_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=val_dataset.column_names,
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./results/{model_label.lower().replace('-', '_')}",
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        learning_rate=3e-4,
        warmup_ratio=0.1,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=5,
        weight_decay=0.01,
        predict_with_generate=True,
        report_to="none",
        seed=42,
        data_seed=42,
        fp16=torch.cuda.is_available(),
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()

    # Collect logs by epoch so training and validation metrics appear in the same row
    epoch_logs = {}

    for log in trainer.state.log_history:
        if "epoch" not in log:
            continue

        epoch = round(float(log["epoch"]), 2)

        if epoch not in epoch_logs:
            epoch_logs[epoch] = {
                "Model": model_label,
                "Epoch": epoch,
                "Training Loss": None,
                "Validation Loss": None,
                "Gradient Norm": None,
                "Learning Rate": None,
            }

        if "loss" in log:
            epoch_logs[epoch]["Training Loss"] = log["loss"]

        if "eval_loss" in log:
            epoch_logs[epoch]["Validation Loss"] = log["eval_loss"]

        if "grad_norm" in log:
            epoch_logs[epoch]["Gradient Norm"] = log["grad_norm"]

        if "learning_rate" in log:
            epoch_logs[epoch]["Learning Rate"] = log["learning_rate"]

    metric_rows = list(epoch_logs.values())

    return {
        "model": model,
        "tokenizer": tokenizer,
        "trainer": trainer,
        "loss_rows": metric_rows,
    }

## 7. Train T5-small

In [ ]:
trained_models = {}
all_loss_rows = []

t5_result = train_model(
    model_label="T5-small",
    model_name="google-t5/t5-small"
)

trained_models["T5-small"] = t5_result
all_loss_rows.extend(t5_result["loss_rows"])

## 8. Train FLAN-T5-small

In [ ]:
flan_result = train_model(
    model_label="FLAN-T5-small",
    model_name="google/flan-t5-small"
)

trained_models["FLAN-T5-small"] = flan_result
all_loss_rows.extend(flan_result["loss_rows"])


# 9. Loss Comparison Table

In [ ]:

loss_comparison = pd.DataFrame(all_loss_rows)
loss_comparison

metrics_df = pd.DataFrame(all_loss_rows)

metrics_df = metrics_df.sort_values(["Model", "Epoch"]).reset_index(drop=True)

metrics_df

# 10. Define a Function for Yoda-Style Generation

This cell defines a helper function that takes a normal English sentence and a model name, applies the same task prefix used during training, generates a Yoda-style sentence, and decodes the model output back into readable text.


In [ ]:
def generate_yoda(sentence, model_label, max_length=64):
    model = trained_models[model_label]["model"]
    tokenizer = trained_models[model_label]["tokenizer"]

    input_text = "translate English to Yoda style: " + sentence
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
    ).to(model.device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# 11. Test the Models on Edge Cases

This cell evaluates the models qualitatively on more difficult inputs, such as very short sentences, questions, negation, named entities, technical terminology, punctuation, emojis, already Yoda-like input, and long sentences. This helps reveal strengths and weaknesses that may not be visible from validation loss alone.


In [ ]:
from IPython.display import display, HTML

edge_cases = [
    {
        "Category": "Very short input",
        "Input": "Run."
    },
    {
        "Category": "Single word",
        "Input": "Learning."
    },
    {
        "Category": "Question",
        "Input": "Why is the model producing incorrect results?"
    },
    {
        "Category": "Yes-or-no question",
        "Input": "Can you finish the assignment today?"
    },
    {
        "Category": "Negation",
        "Input": "I do not understand this lesson."
    },
    {
        "Category": "Contraction",
        "Input": "I can't complete the project without help."
    },
    {
        "Category": "Imperative command",
        "Input": "Please submit the report before midnight."
    },
    {
        "Category": "Multiple clauses",
        "Input": (
            "Although the dataset was small, the model performed well "
            "because we carefully cleaned the training examples."
        )
    },
    {
        "Category": "Numbers and percentages",
        "Input": "The model achieved 92.5 percent accuracy after 10 epochs."
    },
    {
        "Category": "Named entities",
        "Input": "Luke Skywalker traveled to New York with Professor Smith."
    },
    {
        "Category": "Technical terminology",
        "Input": (
            "The transformer uses self-attention to create contextual "
            "representations of each token."
        )
    },
    {
        "Category": "Repeated words",
        "Input": "The model is very, very, very difficult to train."
    },
    {
        "Category": "Strong punctuation",
        "Input": "Stop! This experiment is extremely dangerous!"
    },
    {
        "Category": "Quoted speech",
        "Input": 'The student said, "I will finish the experiment tomorrow."'
    },
    {
        "Category": "Already Yoda-like",
        "Input": "Powerful you have become, young learner."
    },
    {
        "Category": "Unusual characters",
        "Input": "AI, robots, and data—are they the future? 🤖"
    },
    {
        "Category": "Long input",
        "Input": (
            "The research team trained several language models, evaluated "
            "them on multiple datasets, compared their validation losses, "
            "and selected the model that produced the most consistent "
            "results across all experiments."
        )
    },
]

generation_rows = []

for example in edge_cases:
    row = {
        "Category": example["Category"],
        "Input": example["Input"],
    }

    for model_label in trained_models:
        try:
            row[model_label] = generate_yoda(
                example["Input"],
                model_label
            )
        except Exception as error:
            row[model_label] = f"ERROR: {error}"

    generation_rows.append(row)

generation_comparison = pd.DataFrame(generation_rows)

# Prevent long text from being truncated in the notebook table.
pd.set_option("display.max_colwidth", None)

from IPython.display import display, HTML

# Keep one comparison table with both model outputs side by side
comparison_table = generation_comparison[
    ["Category", "Input", "T5-small", "FLAN-T5-small"]
]


html_table = comparison_table.to_html(
    index=False,
    escape=False,
    classes="ltr-table"
)


html = f"""
<div dir="ltr" style="
    direction: ltr;
    text-align: left;
    width: 100%;
">
<style>
.ltr-table {{
    direction: ltr !important;
    text-align: left !important;
    border-collapse: collapse;
    width: 100%;
    table-layout: fixed;
    font-size: 13px;
}}

.ltr-table th,
.ltr-table td {{
    direction: ltr !important;
    text-align: left !important;
    border: 1px solid #444;
    padding: 8px;
    vertical-align: top;
    white-space: normal;
    word-wrap: break-word;
}}

.ltr-table th {{
    background-color: #222;
    color: white;
}}
</style>

{html_table}
</div>
"""

display(HTML(html))

# 11. Plot the Loss


In [ ]:
import matplotlib.pyplot as plt

plot_df = metrics_df.copy()

metrics_to_plot = [
    "Training Loss",
    "Validation Loss",
    "Gradient Norm",
    "Learning Rate",
]

for metric in metrics_to_plot:
    plt.figure(figsize=(7, 4))

    for model_name in plot_df["Model"].unique():
        model_data = plot_df[plot_df["Model"] == model_name]
        model_data = model_data.dropna(subset=[metric])

        if len(model_data) == 0:
            continue

        plt.plot(
            model_data["Epoch"],
            model_data[metric],
            marker="o",
            label=model_name,
        )

    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.title(f"{metric} Comparison")
    plt.legend()
    plt.grid(True)
    plt.show()

## To make it a bit more fun ^^

In [ ]:
!pip install ipywidgets -q

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

input_box = widgets.Text(
    value="",
    placeholder="Type an English sentence here...",
    description="You:",
    layout=widgets.Layout(width="70%")
)

button = widgets.Button(
    description="Transform",
    button_style="success",
    layout=widgets.Layout(width="120px")
)

output_area = widgets.Output()

def on_button_click(b):
    with output_area:
        output_area.clear_output()

        sentence = input_box.value.strip()

        if sentence == "":
            print("Please type a sentence first.")
            return

        response = generate_yoda(sentence, "FLAN-T5-small")

        display(Markdown(f"**YodaBot:** {response}"))

button.on_click(on_button_click)

chat_row = widgets.HBox(
    [input_box, button],
    layout=widgets.Layout(
        width="100%",
        align_items="center",
        padding="60px",
        background_color="#ffffff",
    )
)

display(chat_row, output_area)